# 3.3 LCEL — LangChain Expression Language

## Playground Notebook

LCEL is LangChain's **declarative composition framework**. It lets you build complex LLM pipelines by chaining components with the pipe `|` operator.

| Topic | What You'll Learn |
|-------|-------------------|
| **The Pipe Operator** | Building chains with `prompt \| model \| parser` |
| **Runnable Protocol** | `invoke()`, `stream()`, `batch()` — the universal interface |
| **RunnablePassthrough** | Forwarding and enriching data through chains |
| **RunnableParallel** | Executing multiple branches simultaneously |
| **RunnableLambda** | Injecting custom Python functions into chains |
| **RunnableBranch** | Conditional routing (if/else for chains) |
| **Fallbacks & Retry** | Automatic error recovery and resilience |
| **Streaming & Batching** | First-class streaming and parallel batch processing |

Think of LCEL as the **plumbing system** — individual components are useful on their own, but LCEL connects them into a pipeline where data flows seamlessly and streaming, async, batching, and observability come for free.

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [2]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. The Runnable Protocol — Foundation of LCEL

Every component in LCEL implements the **Runnable protocol** — a universal interface that makes composition possible.

| Method | Description | Use Case |
|--------|-------------|----------|
| `invoke()` | Process single input synchronously | Standard request-response |
| `stream()` | Yield results incrementally | Real-time chat UIs |
| `batch()` | Process multiple inputs in parallel | Bulk processing |
| `ainvoke()` | Async invoke | High-concurrency servers |
| `astream()` | Async streaming | Async chat apps |

**Critical insight:** When you compose Runnables with `|`, the resulting chain is *itself* a Runnable. So you can `invoke()`, `stream()`, or `batch()` the entire chain.

### Input/Output Types

| Component | Input | Output |
|-----------|-------|--------|
| `ChatPromptTemplate` | `dict` (variables) | `ChatPromptValue` (messages) |
| `ChatModel` | messages | `AIMessage` |
| `StrOutputParser` | `AIMessage` | `str` |
| `JsonOutputParser` | `AIMessage` | `dict` |
| `RunnablePassthrough` | any | same as input |
| `RunnableLambda` | any | any (depends on function) |

### Experiment 1A: The Fundamental Chain Pattern

In [3]:
# The fundamental pattern: prompt | model | parser
#
# Data flow:
#   {variables} → ChatPromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant."),
    ("human", "{question}")
])

parser = StrOutputParser()

# Build the chain with the pipe operator
chain = prompt | llm | parser

# The chain is itself a Runnable
print(f"Chain type: {type(chain).__name__}")

# invoke() — single input, full response
result = chain.invoke({"question": "What is LCEL in LangChain? 2 sentences."})

print(f"Result type: {type(result).__name__}")  # str, not AIMessage
display(Markdown(result))

Chain type: RunnableSequence
Result type: TextAccessor


In LangChain, LCEL stands for Language Chain, a data structure used to represent a hierarchical chain of language models. It is a linked list data structure that allows for efficient navigation and traversal of a sequence of language models.

### Experiment 1B: The Runnable Protocol in Action — `invoke`, `stream`, `batch`

In [4]:
# The SAME chain supports all three modes automatically

# --- invoke: single input ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = chain.invoke({"question": "What is the Runnable protocol? One sentence."})
print(result)

invoke() — Single input, full response
The Runnable protocol is a Java-based protocol that enables Java bytecode to run in a sandboxed environment, allowing for secure and isolated execution of Java code, while also providing features like memory management and access control.


In [5]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in chain.stream({"question": "Name 3 LCEL primitives. Brief."}):
    print(chunk, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are 3 LCEL primitives:

1. `add`
2. `sub`
3. `mul`


In [6]:
# --- batch: multiple inputs in parallel ---
print("=" * 50)
print("batch() — Process multiple inputs at once")
print("=" * 50)

questions = [
    {"question": "What is RunnablePassthrough? One sentence."},
    {"question": "What is RunnableParallel? One sentence."},
    {"question": "What is RunnableLambda? One sentence."},
]

start = time.time()
results = chain.batch(questions)
elapsed = time.time() - start

for q, r in zip(questions, results):
    print(f"Q: {q['question']}")
    print(f"A: {r}\n")

print(f"Processed {len(questions)} inputs in {elapsed:.2f}s (parallel!)")

batch() — Process multiple inputs at once
Q: What is RunnablePassthrough? One sentence.
A: Runnables Passthrough is a technique in Java that allows a Runnable to be used in a way similar to a traditional method call, enabling the use of static methods, lambda expressions, and method references with Runnable objects.

Q: What is RunnableParallel? One sentence.
A: RunnableParallel is a software framework that allows developers to write parallel and concurrent code in a sequential and easy-to-understand programming style, making it accessible to developers who are new to parallel programming.

Q: What is RunnableLambda? One sentence.
A: RunnableLambda is a Java 8 feature that allows a lambda expression to be treated as a Runnable, enabling it to be executed as a separate thread, typically in a thread pool or executor service.

Processed 3 inputs in 2.20s (parallel!)


---

## 2. Extending the Chain — Adding More Components

Each additional component in the pipe adds a processing step. The chain remains a single Runnable.

```
chain = prompt | model | parser | post_processor | formatter
```

### Experiment 2A: Extending with Post-Processing

In [7]:
# Add a post-processing step using RunnableLambda
def add_disclaimer(text):
    return f"{text}\n\n--- Generated by {MODEL} via LCEL ---"

# Extended chain: prompt → model → parse → post-process
extended_chain = prompt | llm | StrOutputParser() | RunnableLambda(add_disclaimer)

result = extended_chain.invoke({"question": "What is LangChain? One sentence."})
print(result)

LangChain is a Rust-based framework that enables developers to build scalable, decentralized, and interoperable blockchain applications by providing a set of libraries and tools for data management, querying, and linking blockchain data.

--- Generated by llama3.2:3b via LCEL ---


### Experiment 2B: Chain of Chains — Composing Multiple Chains

In [8]:
# Chain 1: Generate a technical explanation
explain_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a technical writer. 2-3 sentences."),
        ("human", "Explain: {topic}")
    ])
    | llm
    | StrOutputParser()
)

# Chain 2: Simplify for a child
simplify_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Simplify this for a 10-year-old. Use an analogy. 2 sentences."),
        ("human", "{text}")
    ])
    | llm
    | StrOutputParser()
)

# Run in sequence — output of chain 1 feeds into chain 2
topic = "LCEL pipe operator"

print("Step 1 — Technical Explanation:")
print("=" * 50)
explanation = explain_chain.invoke({"topic": topic})
display(Markdown(explanation))

print("\nStep 2 — Simplified:")
print("=" * 50)
simple = simplify_chain.invoke({"text": explanation})
display(Markdown(simple))

Step 1 — Technical Explanation:


The `LCEL` pipe operator is a Perl 5.12+ feature that allows the extraction of the last element of an array. It is used in conjunction with the `@` sign to denote an array. The syntax is `LCEL @array`, which returns the last element of the array, or an empty string if the array is empty.


Step 2 — Simplified:


Imagine you have a big box of crayons. The `LCEL` pipe operator is like a special tool that helps you pick the last crayon out of the box, so you can color with it.

---

## 3. LCEL Primitive 1: RunnablePassthrough — Pass Data Through

`RunnablePassthrough` passes its input through **unchanged** to the next component. Seems trivial, but it's essential for preserving data when you need it downstream.

**`RunnablePassthrough.assign()`** — keeps the original input AND adds new keys. Crucial for enriching data as it flows.

### Experiment 3A: RunnablePassthrough Basics

In [9]:
# RunnablePassthrough — input passes through unchanged
passthrough = RunnablePassthrough()

result = passthrough.invoke("Hello, I pass right through!")
print(f"Input:  'Hello, I pass right through!'")
print(f"Output: '{result}'")
print(f"Same?   {result == 'Hello, I pass right through!'}")

Input:  'Hello, I pass right through!'
Output: 'Hello, I pass right through!'
Same?   True


### Experiment 3B: RunnablePassthrough.assign() — Enrich Data

In [10]:
# assign() keeps original input AND adds new computed keys

# Simulate: given a question, add a "word_count" and "uppercased" field
enricher = RunnablePassthrough.assign(
    word_count=RunnableLambda(lambda x: len(x["question"].split())),
    uppercased=RunnableLambda(lambda x: x["question"].upper())
)

result = enricher.invoke({"question": "What is LCEL in LangChain?"})

print("Input:  {'question': 'What is LCEL in LangChain?'}")
print(f"Output: {result}")
print()
print("Original 'question' is preserved, plus new keys added!")

Input:  {'question': 'What is LCEL in LangChain?'}
Output: {'question': 'What is LCEL in LangChain?', 'word_count': 5, 'uppercased': 'WHAT IS LCEL IN LANGCHAIN?'}

Original 'question' is preserved, plus new keys added!


### Experiment 3C: assign() in a Real Chain — Adding Context

In [11]:
# Practical use: enrich input with LLM-generated context before the main prompt

# Step 1: Generate keywords from the question
keyword_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Extract 3 keywords from this question. Comma-separated, no extra text."),
        ("human", "{question}")
    ])
    | llm
    | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add keywords
enriched_chain = RunnablePassthrough.assign(
    keywords=keyword_chain
)

# Step 3: Use both in the final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question. Focus on these keywords: {keywords}"),
    ("human", "{question}")
])

full_chain = enriched_chain | final_prompt | llm | StrOutputParser()

result = full_chain.invoke({"question": "How does LCEL handle streaming in chains?"})

print("Result:")
display(Markdown(result))

Result:


I couldn't find any information on "LCEL" being a well-known technology or platform related to streaming in chains. It's possible that LCEL is a custom or proprietary solution, or it may be a misspelling or variation of a different term.

If you could provide more context or clarify what LCEL is, I'll do my best to provide a helpful answer.

---

## 4. LCEL Primitive 2: RunnableParallel — Execute Branches Simultaneously

`RunnableParallel` takes a single input and routes it to **multiple Runnables that execute in parallel**. Results are collected into a dictionary.

```
Input
  ├──→ Branch A → result_a
  ├──→ Branch B → result_b
  └──→ Branch C → result_c
  ▼
{"a": result_a, "b": result_b, "c": result_c}
```

**Dictionary shorthand:** `{"key": runnable}` is automatically treated as `RunnableParallel`.

### Experiment 4A: Basic RunnableParallel

In [12]:
# Two branches process the same input in parallel

summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in one sentence."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

keywords_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List 3 keywords. Comma-separated only."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# RunnableParallel — both execute at the same time
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain
)

start = time.time()
result = parallel_chain.invoke({"topic": "LangChain Expression Language (LCEL)"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (both branches ran in parallel!)\n")
print(f"Summary:  {result['summary']}")
print(f"Keywords: {result['keywords']}")

Completed in 0.96s (both branches ran in parallel!)

Summary:  LangChain is an open-source Expression Language (EL) that enables developers to define and execute complex data pipelines, workflows, and business logic in a declarative and efficient manner, using a simple and expressive syntax.
Keywords: LangChain Expression Language (LCEL)


### Experiment 4B: Dictionary Shorthand (Most Common Pattern)

In [13]:
# Dictionary shorthand — cleaner syntax for RunnableParallel
# This is how it's used most often in real LCEL chains

analyze_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are given a topic with its summary and keywords.\n"
     "Summary: {summary}\n"
     "Keywords: {keywords}\n"
     "Write a brief analysis (2 sentences)."),
    ("human", "Analyze this topic.")
])

# Chain: parallel branches → combine into prompt → model → parse
full_chain = (
    {"summary": summary_chain, "keywords": keywords_chain}  # dict shorthand = RunnableParallel
    | analyze_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"topic": "The Runnable protocol in LangChain"})
display(Markdown(result))

The Runnable protocol in LangChain appears to be a significant development in the field of decentralized applications (dApps) on blockchain technology, offering a scalable and secure platform for developers to build and deploy dApps. By providing a standardized framework for executing smart contracts and interacting with blockchain data, the protocol has the potential to increase adoption and usage of blockchain-based applications, thereby driving the growth of the blockchain ecosystem.

### Experiment 4C: RunnableParallel + RunnablePassthrough (The RAG Pattern)

This is the most important LCEL composition — used in every RAG application.

```
Input (question)
  ├──→ "Retriever" → context (documents)
  └──→ RunnablePassthrough → question (preserved)
  ▼
Prompt (combines context + question) → Model → Parser → Answer
```

In [14]:
# Simulated RAG pattern — no vector DB needed, just the LCEL structure

# Simulate a retriever with a function
knowledge_base = {
    "lcel": "LCEL is LangChain's declarative composition framework using the pipe operator.",
    "runnable": "The Runnable protocol provides invoke(), stream(), and batch() on every component.",
    "passthrough": "RunnablePassthrough forwards input unchanged, assign() adds new keys.",
    "parallel": "RunnableParallel executes multiple branches simultaneously.",
}

def fake_retriever(query: str) -> str:
    """Simulate document retrieval by keyword matching."""
    query_lower = query.lower()
    matched = [doc for key, doc in knowledge_base.items() if key in query_lower]
    return " ".join(matched) if matched else "No relevant documents found."

# RAG chain using the dictionary shorthand pattern
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question based ONLY on this context:\n\n{context}\n\n"
     "If the context doesn't contain the answer, say so."),
    ("human", "{question}")
])

rag_chain = (
    {"context": RunnableLambda(lambda x: fake_retriever(x["question"])),
     "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"])}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test with questions
questions = [
    "What is LCEL?",
    "How does RunnableParallel work?",
    "What is the weather today?",  # Not in knowledge base
]

for q in questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    answer = rag_chain.invoke({"question": q})
    print(f"A: {answer}")


Q: What is LCEL?
A: LCEL is LangChain's declarative composition framework using the pipe operator.

Q: How does RunnableParallel work?
A: RunnableParallel executes multiple branches simultaneously.

Q: What is the weather today?
A: No relevant documents found.


---

## 5. LCEL Primitive 3: RunnableLambda — Custom Functions as Runnables

`RunnableLambda` wraps any Python function into a Runnable, making it composable in LCEL chains.

Use it for: data transformation, validation, formatting, external API calls, logging — anything that doesn't fit a built-in component.

### Experiment 5A: Basic RunnableLambda

In [15]:
# Wrap any function as a Runnable
def word_count(text: str) -> dict:
    """Count words and characters."""
    return {
        "text": text,
        "words": len(text.split()),
        "chars": len(text)
    }

counter = RunnableLambda(word_count)

# It supports the full Runnable interface
result = counter.invoke("LCEL makes chains composable and powerful")
print(f"invoke(): {result}")

# batch() works too
results = counter.batch(["Hello world", "LCEL is great", "Three word sentence"])
print(f"\nbatch(): {results}")

invoke(): {'text': 'LCEL makes chains composable and powerful', 'words': 6, 'chars': 41}

batch(): [{'text': 'Hello world', 'words': 2, 'chars': 11}, {'text': 'LCEL is great', 'words': 3, 'chars': 13}, {'text': 'Three word sentence', 'words': 3, 'chars': 19}]


### Experiment 5B: RunnableLambda in a Chain — Custom Post-Processing

In [16]:
# Custom functions as chain steps

def format_as_bullet_points(text: str) -> str:
    """Convert newline-separated text into bullet points."""
    lines = [line.strip() for line in text.strip().split("\n") if line.strip()]
    return "\n".join(f"  * {line}" for line in lines)

def add_header(text: str) -> str:
    """Add a formatted header."""
    return f"{'=' * 40}\n  LCEL CHEAT SHEET\n{'=' * 40}\n{text}\n{'=' * 40}"

# Chain: prompt → model → parse → format → header
formatted_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List each item on its own line. No numbering."),
        ("human", "List the 5 LCEL primitives and what each does.")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_bullet_points)
    | RunnableLambda(add_header)
)

result = formatted_chain.invoke({})
print(result)

  LCEL CHEAT SHEET
  * Here are the 5 LCEL primitives:
  * L - Locate (returns the location of a value in memory)
  * C - Compare (compares two values and returns a boolean result)
  * E - Examine (allows examining the value of a variable without changing its value)
  * L - Load (loads a value from memory into a register)
  * C - Constant (returns the value of a constant expression)


### Experiment 5C: The @chain Decorator — Cleaner Syntax

In [17]:
from langchain_core.runnables import chain as chain_decorator

# The @chain decorator turns a function into a Runnable
# Cleaner than wrapping with RunnableLambda

@chain_decorator
def analyze_and_respond(input_dict: dict) -> str:
    """A custom Runnable that does multi-step processing."""
    question = input_dict["question"]

    # Step 1: Classify the question type
    classify_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "Classify this as 'technical' or 'general'. One word only."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )
    category = classify_chain.invoke({"q": question}).strip().lower()

    # Step 2: Answer based on classification
    if "technical" in category:
        style = "a precise software engineer with code examples"
    else:
        style = "a friendly teacher using simple analogies"

    answer_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You are {style}. Keep answers to 2-3 sentences."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )

    answer = answer_chain.invoke({"style": style, "q": question})
    return f"[{category.upper()}] {answer}"

# Use it like any other Runnable
result = analyze_and_respond.invoke({"question": "How does the pipe operator work in LCEL?"})
print(result)

print()

result2 = analyze_and_respond.invoke({"question": "Why is the sky blue?"})
print(result2)

[TECHNICAL] In L CEL, the pipe operator (`|`) is used to create a conditional expression that evaluates to one of two values. It is often used in the `if` statement to replace the traditional `if-then-else` syntax. Here is an example:

```c
if (a | b) then a;
```

In this example, if `a` is true, the expression evaluates to `a` (i.e., `a` is assigned the value `a`).

[GENERAL] Let me explain it in a simple way. The sky appears blue because of a thing called light. When sunlight enters Earth's atmosphere, it scatters, or bounces, off tiny particles in the air, and that's what makes the sky look blue to our eyes!


---

## 6. LCEL Primitive 4: RunnableBranch — Conditional Routing

`RunnableBranch` routes input to **different Runnables based on conditions**. Like an `if/elif/else` for chains.

```
Input
  ▼
RunnableBranch
  ├── condition_1 is True → chain_a
  ├── condition_2 is True → chain_b
  └── default → chain_c
```

### Experiment 6A: Basic Conditional Routing

In [18]:
# Different chains for different input types

technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a precise engineer. Use technical terminology. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Use metaphors and vivid language. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

# Route based on keywords in the question
router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["code", "api", "function", "implement"]),
     technical_chain),
    (lambda x: any(w in x["question"].lower() for w in ["story", "imagine", "creative", "poem"]),
     creative_chain),
    general_chain  # default
)

# Test with different types of questions
test_questions = [
    "How do I implement a chain in code?",
    "Imagine LCEL as a story character.",
    "What is the weather like today?",
]

for q in test_questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    result = router.invoke({"question": q})
    print(result)


Q: How do I implement a chain in code?
[TECHNICAL] To implement a chain in code, you can utilize a data structure such as a linked list, where each node represents a link in the chain, and each node contains a reference to the next node in the sequence, effectively creating a linear data structure with no branches. In a programming language like C++, you can implement a chain using a struct or class that contains a pointer to the next node in the sequence, allowing you to dynamically add or remove nodes from the chain.

Example:
```cpp
struct Node {
    int value;
    Node* next;
};

// Function to add a new node to the end of the chain
void append(Node** head, int value) {
    Node* newNode = new Node();
    newNode->value = value;
    newNode->next = nullptr;

    if (*head == nullptr) {
        *head = newNode;
    } else {
        Node* current = *head;
        while (current->next != nullptr) {
            current = current->next;
        }
        current->next = newNode;
    }


### Experiment 6B: LLM-Powered Routing (Classify then Route)

In [19]:
# Use the LLM itself to classify, then route to the right chain

# Step 1: Classify the question
classifier = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Classify this question into exactly ONE category: technical, creative, or general.\n"
         "Respond with the category name only. One word."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add classification
classified_chain = RunnablePassthrough.assign(
    category=classifier
)

# Step 3: Route based on classification
smart_router = RunnableBranch(
    (lambda x: "technical" in x["category"].lower(), technical_chain),
    (lambda x: "creative" in x["category"].lower(), creative_chain),
    general_chain
)

# Full pipeline: classify → route → respond
smart_chain = classified_chain | smart_router

test_qs = [
    "How does the Runnable protocol handle async execution?",
    "Write me a poem about data pipelines.",
    "What should I have for lunch?",
]

for q in test_qs:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    # Show classification
    classified = classified_chain.invoke({"question": q})
    print(f"Category: {classified['category'].strip()}")
    print(f"{'=' * 50}")
    result = smart_router.invoke(classified)
    print(result)


Q: How does the Runnable protocol handle async execution?
Category: technical
[TECHNICAL] The Runnable protocol utilizes a thread pool-based approach to manage asynchronous execution, utilizing a combination of Java ForkJoinPool and ExecutorService to execute tasks concurrently, while also employing a callback mechanism to notify the application of task completion. The Runnable protocol also leverages Java's AQS (Atomic-Quasi-Atomic) synchronization mechanisms to ensure that tasks are executed in a thread-safe and ordered manner.

Q: Write me a poem about data pipelines.
Category: In realms of code, where data reigns
A kingdom built of lines and pains
A pipeline born, of logic's might
A journey through the digital night

It starts with sources, pure and bright
A flow of facts, in morning light
Through processing steps, it's refined
And cleansed of error, left behind

The transformations, a wondrous spin
A dance of data, locked within
The aggregations, a sum so fine
A tale of numbers, 

---

## 7. LCEL Primitive 5: Fallbacks — Error Recovery

`.with_fallbacks()` wraps a Runnable with backup alternatives. If the primary fails, it automatically tries the next one.

```python
robust_model = primary_model.with_fallbacks([backup_model_1, backup_model_2])
```

### Experiment 7A: Fallback Chain

In [20]:
# Simulate a failing model with a function that always raises

def unreliable_model(input_text):
    """Simulates an LLM that fails."""
    raise ConnectionError("Model API is down!")

# Primary: the unreliable model
primary = RunnableLambda(unreliable_model)

# Fallback: our reliable Ollama model
fallback_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Combine with fallback
resilient_chain = primary.with_fallbacks([fallback_chain])

# The primary will fail, but the fallback catches it
result = resilient_chain.invoke({"question": "What is LCEL?"})

print(f"Primary failed but fallback succeeded!")
print(f"Result: {result}")

Primary failed but fallback succeeded!
Result: I couldn't find any information on "LCEL". It's possible that it's an acronym or abbreviation for a specific term or concept that I'm not aware of. Can you please provide more context or information about what LCEL refers to?


### Experiment 7B: Retry Logic with `.with_retry()`

In [21]:
# .with_retry() adds automatic retries with exponential backoff

attempt_count = 0

def flaky_function(x):
    """Fails twice, then succeeds on the third attempt."""
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        print(f"  Attempt {attempt_count}: FAILED")
        raise ConnectionError(f"Transient error on attempt {attempt_count}")
    print(f"  Attempt {attempt_count}: SUCCESS")
    return f"Result after {attempt_count} attempts: {x}"

# Wrap with retry logic
retry_runnable = RunnableLambda(flaky_function).with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

print("Invoking flaky function with retry...")
result = retry_runnable.invoke("hello")
print(f"\nFinal result: {result}")

Invoking flaky function with retry...
  Attempt 1: FAILED
  Attempt 2: FAILED
  Attempt 3: SUCCESS

Final result: Result after 3 attempts: hello


### Experiment 7C: Combining Retry + Fallback (Maximum Resilience)

The most robust pattern:
```python
resilient = (
    primary.with_retry(stop_after_attempt=2)
    .with_fallbacks([
        fallback.with_retry(stop_after_attempt=2)
    ])
)
```

In [22]:
# Demonstrate the pattern conceptually

# In production with multiple providers, this would look like:
# resilient_llm = (
#     ChatOpenAI(model="gpt-4o").with_retry(stop_after_attempt=2)
#     .with_fallbacks([
#         ChatAnthropic(model="claude-3-5-sonnet").with_retry(stop_after_attempt=2),
#         ChatOllama(model="llama3.2:3b")  # local fallback, no retry needed
#     ])
# )

# With our local setup — retry the same model
resilient_llm = llm.with_retry(stop_after_attempt=3)

resilient_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Be concise. One sentence."),
        ("human", "{question}")
    ])
    | resilient_llm
    | StrOutputParser()
)

result = resilient_chain.invoke({"question": "What makes LCEL resilient?"})
print(result)

LCEL, a language model, is resilient due to its ability to process and generate human-like text based on a large corpus of data, allowing it to adapt and learn from various sources and minimize the impact of noise or inconsistencies in the input.


---

## 8. Streaming Deep Dive

When you call `.stream()` on an LCEL chain, every component streams its output to the next as tokens become available.

| Component | Streaming Behavior |
|-----------|-----------------------|
| Chat Model | Yields tokens as generated |
| StrOutputParser | Passes through each token immediately |
| JsonOutputParser | Accumulates until valid JSON, then yields |
| RunnablePassthrough | Passes through immediately |
| RunnableLambda | Depends on function (generators stream) |
| Retriever | Returns complete results (no partial streaming) |

### Experiment 8A: Streaming Through a Chain — Token Timing

In [23]:
# Compare: invoke (wait for all) vs stream (token by token)

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

question = {"question": "List 3 benefits of Langchain Expression Language in langchain. One sentence each."}

# --- invoke: wait for complete response ---
print("=" * 50)
print("invoke() — User waits for full response")
print("=" * 50)
start = time.time()
result = chain.invoke(question)
elapsed = time.time() - start
print(result)
print(f"\nTotal wait: {elapsed:.2f}s")

invoke() — User waits for full response
Here are three benefits of Langchain Expression Language in Langchain:

1. The Langchain Expression Language provides a standardized and simple syntax for specifying complex data processing tasks and workflows.
2. It enables users to focus on the logic and intent behind their tasks without worrying about the intricacies of the underlying data processing pipelines.
3. The language is extensible and can be easily modified or extended by users to accommodate specific domain requirements or new data formats.

Total wait: 1.46s


In [24]:
# --- stream: tokens arrive immediately ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)
start = time.time()
first_token_time = None
token_count = 0

for chunk in chain.stream(question):
    if first_token_time is None:
        first_token_time = time.time() - start
    print(chunk, end="", flush=True)
    token_count += 1

elapsed = time.time() - start
print(f"\n\nTime to first token: {first_token_time:.2f}s")
print(f"Total time: {elapsed:.2f}s")
print(f"Chunks received: {token_count}")
print("\nUsers perceive streaming as faster even when total time is similar!")

stream() — Tokens arrive as generated
Here are three benefits of Langchain Expression Language in Langchain:

1. Langchain Expression Language allows for more flexible and efficient expression of complex tasks and workflows, making it easier to integrate and compose multiple models and services.
2. It provides a standardized and Pythonic API for defining and executing complex tasks, making it easier for developers to build and maintain Langchain models.
3. Langchain Expression Language enables the creation of reusable and modular code that can be easily shared and combined with other models and services, promoting collaboration and reducing development time.

Time to first token: 0.11s
Total time: 1.64s
Chunks received: 110

Users perceive streaming as faster even when total time is similar!


---

## 9. Batch Processing Deep Dive

`.batch()` processes multiple inputs in **parallel**. Configurable with `max_concurrency`.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_concurrency` | None (unlimited) | Max parallel executions |
| `return_exceptions` | False | Return exceptions instead of raising |

### Experiment 9A: Batch with Timing Comparison

In [25]:
chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Answer in exactly one sentence."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

questions = [
    {"question": "What is LCEL?"},
    {"question": "What is a Runnable?"},
    {"question": "What is RunnablePassthrough?"},
    {"question": "What is RunnableParallel?"},
]

# --- Sequential (invoke one by one) ---
print("Sequential (invoke one by one):")
start = time.time()
sequential_results = [chain.invoke(q) for q in questions]
seq_time = time.time() - start
print(f"  Time: {seq_time:.2f}s")

# --- Batch (all at once) ---
print("\nBatch (parallel):")
start = time.time()
batch_results = chain.batch(questions)
batch_time = time.time() - start
print(f"  Time: {batch_time:.2f}s")

print(f"\nSpeedup: {seq_time/batch_time:.1f}x faster with batch!")

print("\nResults:")
for q, r in zip(questions, batch_results):
    print(f"  Q: {q['question']}")
    print(f"  A: {r}\n")

Sequential (invoke one by one):
  Time: 3.32s

Batch (parallel):
  Time: 2.72s

Speedup: 1.2x faster with batch!

Results:
  Q: What is LCEL?
  A: LCEL is an abbreviation for the Learning English for Career Enhancement, a language course designed to help professionals improve their English language skills for career development and professional communication.

  Q: What is a Runnable?
  A: A Runnable is an interface in Java that is implemented by classes that contain the main method, which is the entry point of a Java application, and is used to identify the class as the primary entry point of an application.

  Q: What is RunnablePassthrough?
  A: RunnablePassthrough is a feature in the Android operating system that allows a child process to inherit the process ID and other characteristics of its parent process, enabling them to run in the same process space as the parent.

  Q: What is RunnableParallel?
  A: RunnableParallel is a high-level programming framework that provides a simpl

In [26]:
# Batch with max_concurrency — respect rate limits
print("Batch with max_concurrency=2:")
start = time.time()
results = chain.batch(questions, config={"max_concurrency": 2})
elapsed = time.time() - start
print(f"  Time: {elapsed:.2f}s (limited to 2 concurrent requests)")
print(f"  Results: {len(results)} answers")

Batch with max_concurrency=2:
  Time: 2.64s (limited to 2 concurrent requests)
  Results: 4 answers


---

## 10. Advanced Pattern: Sequential Processing with Context Accumulation

Build up context progressively — each step has access to everything produced by previous steps.

```
Input → Step 1 (summary) → Step 2 (key points using summary) → Step 3 (report using all above)
```

### Experiment 10A: Multi-Step Analysis with assign()

In [27]:
# Progressive context accumulation using assign()

# Step 1: Summarize the topic
summarize = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in 1-2 sentences."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Extract key points (has access to topic + summary)
extract_points = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Given this summary: {summary}\n\n"
         "List 3 key points about the original topic. One line each."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 3: Generate a verdict (has access to topic + summary + key_points)
verdict = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Based on:\nSummary: {summary}\nKey Points: {key_points}\n\n"
         "Give a one-sentence verdict on the topic's importance."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Chain with progressive context accumulation
analysis_chain = (
    RunnablePassthrough.assign(summary=summarize)         # adds 'summary'
    | RunnablePassthrough.assign(key_points=extract_points)  # adds 'key_points'
    | RunnablePassthrough.assign(verdict=verdict)            # adds 'verdict'
)

result = analysis_chain.invoke({"topic": "LCEL (LangChain Expression Language)"})

print("PROGRESSIVE ANALYSIS")
print("=" * 50)
print(f"\nTopic: {result['topic']}")
print(f"\nSummary:\n  {result['summary']}")
print(f"\nKey Points:\n  {result['key_points']}")
print(f"\nVerdict:\n  {result['verdict']}")

PROGRESSIVE ANALYSIS

Topic: LCEL (LangChain Expression Language)

Summary:
  I couldn't find any information on "LCEL" (LangChain Expression Language). LangChain is a blockchain-based platform, but I couldn't find any details on an "Expression Language" related to it. Can you provide more context or clarify what LCEL is?

Key Points:
  I've found some information on LangChain, but it seems that LCEL is not a widely documented or recognized concept within the LangChain ecosystem.

However, I can provide some context:

LangChain is an open-source blockchain-based platform that enables developers to build decentralized applications (dApps) and integrate them with various blockchain networks.

If you're looking for information on LCEL, it's possible that it's a custom or proprietary implementation of the LangChain platform, or it might be a hypothetical concept.

Unfortunately, I couldn't find any reliable sources that provide detailed information on LCEL. If you could provide more contex

---

## 11. LCEL Best Practices

| Practice | Description |
|----------|-------------|
| Keep chains simple | Start with `prompt \| model \| parser`, add complexity only when needed |
| Name your chains | Use `.with_config(run_name="name")` for easier debugging |
| Use `assign()` for context | Cleaner than manual dictionary construction |
| Prefer dict shorthand | `{"key": runnable}` over explicit `RunnableParallel` |
| Always add fallbacks | LLM APIs fail — plan for it with `.with_fallbacks()` |
| Use retry with backoff | `.with_retry()` handles transient errors gracefully |
| Set `max_concurrency` | Respect API rate limits when using `.batch()` |
| Stream for user-facing apps | Users perceive streaming as faster |
| Test with `invoke()` first | Debug synchronous execution before adding streaming |

### LCEL vs Imperative Code

| Aspect | LCEL (Declarative) | Imperative Code |
|--------|--------------------|-----------------|
| Readability | Excellent for linear/parallel flows | Better for complex branching |
| Streaming | Automatic, first-class | Manual implementation |
| Async | Built-in for every chain | Must rewrite with async/await |
| Batching | One method call | Custom threading |
| Tracing | Automatic (LangSmith) | Manual logging |
| Error handling | Declarative retry/fallback | Try/except blocks |
| Best for | 90% of LLM pipelines | Highly dynamic workflows |

> **Recommendation:** Use LCEL for 90% of your chains. Fall back to imperative code only for highly dynamic workflows — or consider **LangGraph** for those.

---

## 12. Sandbox — Try It Yourself!

In [28]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own LCEL chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different modes
mode = "stream"  # Options: "invoke", "stream", "batch"

# ============================================================

if mode == "invoke":
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": "What is the most important LCEL primitive and why?"})
    display(Markdown(result))

elif mode == "stream":
    print("[STREAM]\n")
    for chunk in my_chain.stream({"question": "What is the most important LCEL primitive and why?"}):
        print(chunk, end="", flush=True)
    print()

elif mode == "batch":
    print("[BATCH]\n")
    results = my_chain.batch([
        {"question": "What is RunnablePassthrough?"},
        {"question": "What is RunnableBranch?"},
        {"question": "What is RunnableLambda?"},
    ])
    for r in results:
        print(f"  {r}\n")

[STREAM]

The most important LCEL (Local Clock Enforcement Level) primitive is LCEL_CLEARTime.

LCEL_CLEARTime is the primitive that clears the local clock, ensuring that the system clock is reset to the time set by the user, rather than being controlled by the system's internal clock. This is crucial to prevent time-related issues, such as:

* System clocks drifting away from the user's desired time
* Incorrect time calculations
* Potential security vulnerabilities related to clock manipulation

By clearing the local clock, LCEL_CLEARTime ensures that the system clock is accurately set and updated, providing a stable and reliable timekeeping mechanism.


---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LCEL** | Declarative composition with the pipe `\|` operator |
| **Runnable Protocol** | Universal interface: `invoke()`, `stream()`, `batch()` on every component |
| **Fundamental Pattern** | `prompt \| model \| parser` — the atom of every chain |
| **RunnablePassthrough** | Forwards data unchanged; `assign()` enriches with new keys |
| **RunnableParallel** | Executes branches simultaneously; dict shorthand `{"key": runnable}` |
| **RunnableLambda** | Wraps any Python function as a composable Runnable |
| **@chain decorator** | Cleaner syntax for custom Runnables |
| **RunnableBranch** | Conditional routing — if/elif/else for chains |
| **with_fallbacks()** | Automatic failover to backup models/chains |
| **with_retry()** | Automatic retries with exponential backoff |
| **Streaming** | Automatic through the entire chain — no extra code needed |
| **Batching** | `.batch()` with `max_concurrency` for parallel + rate-limited processing |
| **RAG Pattern** | `{"context": retriever, "question": passthrough} \| prompt \| model \| parser` |
| **Context Accumulation** | Chain `assign()` calls to progressively build up data |